# StyleWeave Phase 1 B0 (Kaggle)

This notebook **imports repository code**. It does not reimplement ranking.

**Settings:** Internet on. Accelerator **GPU T4** (not P100). Free quota only.

Preflight cells **raise** on coverage, dependency, T4, or disk failure. They do not write `metrics.json`.
A failed cell means later cells must not be treated as a B0 result.

Do **not** upload FashionIQ images. Attach an existing input only:
`https://www.kaggle.com/datasets/hoangkim14/fashion-iq-dataset`

Hugging Face publishing stays **off**. No Qwen. No SigLIP2.

Clone this notebook only after the hardened B0 commit is on GitHub `main`.

In [ ]:
import os, sys, subprocess, pathlib

REPO_DIR = pathlib.Path("/kaggle/working/styleweave-vlm-lab")
REPO_URL = "https://github.com/SumalyaPatnala/styleweave-vlm-lab.git"

if not REPO_DIR.exists():
    token = os.environ.get("GITHUB_TOKEN") or (
        pathlib.Path("/run/secrets/GITHUB_TOKEN").read_text().strip()
        if pathlib.Path("/run/secrets/GITHUB_TOKEN").exists()
        else ""
    )
    clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.check_call(["git", "clone", "--depth", "1", clone_url, str(REPO_DIR)])

sys.path.insert(0, str(REPO_DIR / "src"))
if not (REPO_DIR / "src" / "styleweave" / "eval" / "b0.py").is_file():
    raise RuntimeError(f"Clone failed or incomplete: {REPO_DIR}")
print("repo", REPO_DIR)

In [ ]:
import os, sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.41,<5.3",
    "Pillow",
    "pyyaml",
    "numpy",
])
CACHE = "/kaggle/working/hf-cache"
os.environ["HF_HOME"] = CACHE
os.environ["TRANSFORMERS_CACHE"] = CACHE
os.makedirs(CACHE, exist_ok=True)
print("pinned transformers>=4.41,<5.3; cache", CACHE)

In [ ]:
from styleweave.preflight import PreflightError, check_environment, collect_environment, detect_runtime

print("runtime", detect_runtime())
env_snapshot = collect_environment()
print("python", env_snapshot["python"])
print("torch", env_snapshot["torch"])
print("cuda_available", env_snapshot["cuda_available"], "cuda_version", env_snapshot["cuda_version"])
print("gpu_name", env_snapshot["gpu_name"], "is_t4", env_snapshot["is_t4"])
print("transformers", env_snapshot["transformers"])
print("pillow", env_snapshot["pillow"])
print("clip_api", env_snapshot["clip_api"])
try:
    env = check_environment(require_t4_on_kaggle=True)
except PreflightError as exc:
    raise RuntimeError(f"ENVIRONMENT FAILED; not downloading CLIP; not writing metrics.json: {exc}") from exc
print("environment_ok", env["ok"])

In [ ]:
from styleweave.preflight import PreflightError, check_disk_budget, default_cache_dir, detect_runtime

CACHE = default_cache_dir(detect_runtime(), "/kaggle/working/hf-cache")
try:
    disk = check_disk_budget(
        runtime=detect_runtime(),
        cache_dir=CACHE,
        allow_download=True,
    )
except PreflightError as exc:
    raise RuntimeError(f"DISK FAILED; not downloading CLIP; not writing metrics.json: {exc}") from exc
print("cache_dir", disk["cache_dir"])
print("free_gib", disk["free_gib"])
print("clip_weights_bytes", disk["clip_weights_bytes"])
print("clip_peak_bytes", disk["clip_peak_bytes"])
print("required_free_bytes", disk["required_free_bytes"])
print("weights_cached", disk["weights_cached"])
print("mac_min_free_gib", disk["mac_min_free_gib"])
print("disk_ok", disk["ok"])

In [ ]:
import glob, pathlib
from styleweave.data.fashioniq import discover_fashioniq_layout

layout = None
for root in (pathlib.Path(p) for p in glob.glob("/kaggle/input/*")):
    layout = discover_fashioniq_layout(root)
    if layout:
        break
if layout is None:
    raise RuntimeError(
        "No FashionIQ captions/ + image_splits/ under /kaggle/input. "
        "Attach hoangkim14/fashion-iq-dataset. Do not upload images."
    )
if not pathlib.Path(layout["images_dir"]).is_dir():
    raise RuntimeError(f"No images directory at {layout['images_dir']}")
print(layout)

In [ ]:
from styleweave.config import load_b0_config
from styleweave.eval.b0 import CoverageError, run_b0

cfg = load_b0_config(REPO_DIR / "configs/experiments/phase1_frozen_clip.yaml", root=REPO_DIR)
cfg.captions_dir = str(layout["captions_dir"])
cfg.image_splits_dir = str(layout["image_splits_dir"])
cfg.images_dir = str(layout["images_dir"])
cfg.artifacts_dir = "/kaggle/working/artifacts/phase1_frozen_clip"
cfg.allow_download = True
cfg.hub.enabled = False
cfg.require_complete_gallery = True
cfg.fail_on_missing_reference = True
cfg.fail_on_missing_target = True

try:
    coverage = run_b0(
        cfg,
        dry_run=True,
        images_source="kaggle-input:" + str(layout["root"]),
    )
except CoverageError as exc:
    raise RuntimeError(f"COVERAGE FAILED; not downloading CLIP; not writing metrics.json: {exc}") from exc

if not coverage["coverage"]["ok"]:
    raise RuntimeError("COVERAGE FAILED: " + "; ".join(coverage["coverage"]["errors"]))
print("coverage_ok", True)

In [ ]:
from pathlib import Path
from styleweave.eval.b0 import CoverageError, run_b0
from styleweave.preflight import PreflightError

try:
    payload = run_b0(
        cfg,
        dry_run=False,
        images_source="kaggle-input:" + str(layout["root"]),
        cache_dir=str(CACHE),
    )
except (CoverageError, PreflightError) as exc:
    raise RuntimeError(
        f"B0 FAILED; metrics.json is not this run: {exc}"
    ) from exc

metrics_path = Path(cfg.artifacts_dir) / "metrics.json"
if not metrics_path.is_file():
    raise RuntimeError("B0 finished without metrics.json")
if payload.get("hub_published"):
    raise RuntimeError("Hugging Face publish must stay disabled")
if payload.get("qwen_used") or payload.get("siglip2_used"):
    raise RuntimeError("Qwen/SigLIP2 must stay unused for B0")
print(metrics_path.read_text()[:2500])
print("smoke", payload.get("preflight", {}).get("smoke"))